# 03 — Entrenamiento final del modelo

Entrena el modelo seleccionado en `02_experimentos.ipynb` sobre **todos los datos de train (year ≤ 2023)** y serializa los artefactos para inferencia.

> **No se realizan decisiones de modelado aquí.** La configuración fue fijada en el notebook 02.

In [1]:
import sys
sys.path.insert(0, '..')

import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import mean_squared_error

from src.features import build_features, add_v2_features

SEED = 42
np.random.seed(SEED)

DATA_PATH  = '../data/processed/bgg_games_clean.csv'
MODEL_PATH = '../models/lgbm_final.joblib'

# Hiperparametros definitivos (seleccionados en 02_experimentos.ipynb)
LGB_PARAMS = dict(
    learning_rate    = 0.03,
    num_leaves       = 63,
    n_estimators     = 800,
    min_child_samples= 10,
    subsample        = 0.9,
    colsample_bytree = 0.7,
    reg_alpha        = 0.0,
    reg_lambda       = 0.1,
    random_state     = SEED,
    verbose          = -1,
    n_jobs           = -1,
)

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
print("Config cargada.")

Config cargada.


In [2]:
df       = pd.read_csv(DATA_PATH)
df_train = df[df['year'] <= 2023].reset_index(drop=True)
df_test  = df[df['year'] >= 2024].reset_index(drop=True)

print(f'Train (year <= 2023) : {len(df_train):,} juegos')
print(f'Holdout (2024-2026)  : {len(df_test):,}  juegos  [NO usado en este notebook]')

Train (year <= 2023) : 22,010 juegos
Holdout (2024-2026)  : 2,241  juegos  [NO usado en este notebook]


## Feature engineering

Misma pipeline que en el notebook 02, ahora sobre todo el train:
- Base 80 features: 10 numéricas + top-40 mecánicas + top-30 categorías
- **Ronda 2:** `playtime_ratio`, `players_range`, 10 familias de mecánicas
- `weight_missing` **excluida** (delta < 0.0005 en CV, no aporta señal)
- `designer_enc` **excluida** (mejora CV −0.016 pero empeora holdout +0.089 por shift temporal en diseñadores nuevos — ver análisis en notebook 02)

In [3]:
# Fit: top_mechs y top_cats se calculan SOLO desde train
X_train, y_train, _, top_mechs, top_cats = build_features(df_train)

# Ronda 2: medians se computan desde train y se guardan en artefactos
X_train, medians = add_v2_features(X_train, df_train)
X_train = X_train.drop(columns=['weight_missing'])   # excluida segun analisis

feature_cols = list(X_train.columns)

print(f'X_train shape: {X_train.shape}')
print(f'Features: {len(feature_cols)} columnas')
print(f'Medians  : {medians}')
print(f'top_mechs[0:5]: {top_mechs[:5]}')

C:\Users\ramon\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\preprocessing\_label.py:1007: UserWarning: unknown class(es) ['Acting', 'Action / Event', 'Action Drafting', 'Action Queue', 'Action Retrieval', 'Action Timer', 'Advantage Token', 'Alliances', 'Area-Impulse', 'Auction Compensation', 'Auction: Dexterity', 'Auction: Dutch', 'Auction: Dutch Priority', 'Auction: English', 'Auction: Fixed Placement', 'Auction: Multiple Lot', 'Auction: Once Around', 'Auction: Sealed Bid', 'Auction: Turn Order Until Pass', 'Automatic Resource Growth', 'Betting and Bluffing', 'Bias', 'Bids As Wagers', 'Bingo', 'Bribery', 'Card Play Conflict Resolution', 'Catch the Leader', 'Chaining', 'Chit-Pull System', 'Closed Drafting', 'Closed Economy Auction', 'Command Cards', 'Commodity Speculation', 'Communication Limits', 'Connections', 'Constrained Bidding', 'Contracts', 'Crayon Rail System', 'Critical Hits and Failures', 'Cube Tower', 'Deck Construction', 'Deduction', 'Delayed Purchase',

X_train shape: (22010, 92)
Features: 92 columnas
Medians  : {'max_playtime': 45.0, 'min_playtime': 30.0, 'min_players': 2.0, 'max_players': 4.0}
top_mechs[0:5]: ['Dice Rolling', 'Hand Management', 'Set Collection', 'Variable Player Powers', 'Hexagon Grid']


## Entrenamiento

Un único `fit` sobre todo el train. **Sin CV aquí**: el protocolo de evaluación fue respetado en el notebook 02.

In [4]:
model = lgb.LGBMRegressor(**LGB_PARAMS)
model.fit(X_train, y_train)
print("Modelo entrenado.")
print(f"  n_estimators usados : {model.n_estimators_}")
print(f"  num_leaves          : {model.num_leaves}")

Modelo entrenado.
  n_estimators usados : 800
  num_leaves          : 63


## Sanity check — CV sobre train

Confirma que las métricas reproducen las del notebook 02 (misma semilla, mismo KFold).

In [5]:
res = cross_validate(
    lgb.LGBMRegressor(**LGB_PARAMS), X_train, y_train,
    cv=kf, scoring='neg_root_mean_squared_error', n_jobs=1,
)
fold_rmses = -res['test_score']
rmse_cv_mean = fold_rmses.mean()
rmse_cv_std  = fold_rmses.std()

print(f'RMSE CV (sanity): {rmse_cv_mean:.4f} +/- {rmse_cv_std:.4f}')
print(f'(esperado aprox. 0.5686 +/- 0.0055 del analisis final)')

for i, r in enumerate(fold_rmses):
    print(f'  Fold {i}: {r:.4f}')

RMSE CV (sanity): 0.5686 +/- 0.0055
(esperado aprox. 0.5686 +/- 0.0055 del analisis final)
  Fold 0: 0.5653
  Fold 1: 0.5691
  Fold 2: 0.5614
  Fold 3: 0.5780
  Fold 4: 0.5694


## Serialización de artefactos

In [6]:
import os
os.makedirs('../models', exist_ok=True)

artifacts = {
    # Modelo
    'model':        model,

    # Listas para reproducir binarizacion exacta
    'top_mechs':    top_mechs,
    'top_cats':     top_cats,

    # Medianas de imputacion de ronda-2 (fit en train)
    'medians':      medians,

    # Orden exacto de columnas
    'feature_cols': feature_cols,

    # Metadata
    'train_size':   len(df_train),
    'rmse_cv_mean': float(rmse_cv_mean),
    'rmse_cv_std':  float(rmse_cv_std),
    'lgb_params':   LGB_PARAMS,
    'seed':         SEED,
}

joblib.dump(artifacts, MODEL_PATH, compress=3)
size_kb = os.path.getsize(MODEL_PATH) / 1024
print(f'Artefactos guardados en: {MODEL_PATH}')
print(f'Tamanio: {size_kb:.1f} KB')
print(f'Claves: {list(artifacts.keys())}')

Artefactos guardados en: ../models/lgbm_final.joblib
Tamanio: 1584.3 KB
Claves: ['model', 'top_mechs', 'top_cats', 'medians', 'feature_cols', 'train_size', 'rmse_cv_mean', 'rmse_cv_std', 'lgb_params', 'seed']


## Resumen

| Artefacto | Descripción |
|---|---|
| `model` | LightGBM entrenado sobre 22.010 juegos (year ≤ 2023) |
| `top_mechs` | Lista de top-40 mecánicas (para binarización consistente) |
| `top_cats` | Lista de top-30 categorías |
| `medians` | Medianas de imputación de playtime/players (fit en train) |
| `feature_cols` | Orden exacto de las 92 columnas |

Notebook 04 carga este `.joblib` y reproduce las métricas del holdout.